# E3-A | Semantic Alignment Ablation (SNA-MXH / RQ2)

**Dựa trên notebook `03_E1_Full_Training.ipynb` người dùng cung cấp.** E3-A dùng cùng pinned commit, data, trainer, validation-based early stopping và test như E1. 

- Cấu hình duy nhất: `ALIGN_MODE`, `SEED`. `ALIGN_WEIGHT=None` có nghĩa giữ nguyên `kd_weight` của `lightgcn_plus.yml` trong E1.
- Chọn `graph_only`, `user_only` hoặc `item_only` (3 seeds mỗi loại) để chạy **9 runs mới**. `full` có thể chạy nhưng thông thường **tái sử dụng E1 lightgcn_plus**. `lightgcn` E1 chỉ là baseline tham chiếu, không khẳng định tương đương `graph_only` trong kiến trúc `lightgcn_plus`.
- Code dùng patch **trong RAM** chỉ lọc ba lời gọi InfoNCE tại `cal_loss`; không sửa source pinned RLMRec trên ổ đĩa. Tự kiểm tra AST của source để **fail closed** nếu ba nhánh không còn đúng dạng user / pos-item / neg-item.
- Mọi output trong `experiments/E3/runs/...`; không overwrite E1/E2. `Run All` mỗi cấu hình cần Colab GPU và quyền Google Drive. Checkpoint chọn theo validation, test chỉ dùng đánh giá cuối cùng.
- Notebook chỉ được kiểm tra cấu trúc và biên dịch offline tại thời điểm tạo; **phải smoke-test thực tế trên Colab GPU trước khi giao thành viên**. In `E3A_VERIFICATION: PASS` khi nghiệm thu hết.
**V2 FIX (20/10/2026):** Thay kiểm tra thứ tự `ast.walk()` và bật/tắt InfoNCE theo số thứ tự bằng kiểm tra định danh embedding (`anc_embeds`, `pos_embeds`, `neg_embeds`). Tránh tắt nhầm alignment nếu trình tự AST khác runtime. Dùng **Runtime → Restart session → Run all** khi chuyển từ notebook V1.


In [ ]:
# CELL 1 — ONLY EDIT THE FOLLOWING CONFIG
ALIGN_MODE = "user_only"  # graph_only / user_only / item_only / full
SEED = 2023               # 2023 / 2024 / 2025
ALIGN_WEIGHT = None       # None = inherit EXACT E1 RLMRec-Con kd_weight; numeric only if leader locks a sweep

# Fixed
import os, sys, json, subprocess, math, shutil
from pathlib import Path
import torch
from google.colab import drive

DATASET = "amazon"
MODEL = "lightgcn_plus"
RLMREC_COMMIT = "22413752246de3dee8ab0d509f7f7a8889080f95"
RLMREC_URL = "https://github.com/HKUDS/RLMRec.git"
assert ALIGN_MODE in {"graph_only", "user_only", "item_only", "full"}
assert SEED in (2023, 2024, 2025)
assert ALIGN_WEIGHT is None or (isinstance(ALIGN_WEIGHT, (int, float)) and 0 < ALIGN_WEIGHT < 100)
assert torch.cuda.is_available(), "Enable a GPU runtime in Colab"

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA_DIR = ROOT / 'data/raw/amazon'
REPO = Path('/content/RLMRec_E3')  # independent of E1's /content/RLMRec
assert DATA_DIR.is_dir(), DATA_DIR
WEIGHT_TAG = 'e1_default' if ALIGN_WEIGHT is None else f'{ALIGN_WEIGHT:g}'.replace('.','p')
OUTPUT_DIR = ROOT / 'experiments/E3/runs' / ALIGN_MODE / f'weight_{WEIGHT_TAG}' / f'seed_{SEED}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert not (OUTPUT_DIR/'metrics.json').exists(), f"Finished E3 run exists: {OUTPUT_DIR}; choose another seed/mode."
print('E3 MODE:',ALIGN_MODE,'SEED:',SEED,'WEIGHT:',WEIGHT_TAG,'GPU:',torch.cuda.get_device_name(0))
print('OUTPUT:',OUTPUT_DIR)

In [ ]:
# CELL 2 — PIN SOURCE, LINK DATA, ISOLATE E3 OUTPUT
if not REPO.exists():
    subprocess.run(['git', 'clone', RLMREC_URL, str(REPO)], check=True)
origin = subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()
assert origin.rstrip('/').removesuffix('.git') == RLMREC_URL.rstrip('/').removesuffix('.git'), origin
commit = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if commit != RLMREC_COMMIT:
    status = subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    assert not status, 'Dirty RLMRec E3 source; restart with clean workspace'
    subprocess.run(['git','-C',str(REPO),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',RLMREC_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == RLMREC_COMMIT
assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()

required = ['trn_mat.pkl','val_mat.pkl','tst_mat.pkl','usr_prf.pkl','itm_prf.pkl','usr_emb_np.pkl','itm_emb_np.pkl']
for f in required:
    assert (DATA_DIR/f).is_file(), f'Missing input: {f}'
target = REPO/'data'/DATASET
if not target.exists() and not target.is_symlink(): target.symlink_to(DATA_DIR,target_is_directory=True)
assert target.resolve() == DATA_DIR.resolve(), 'Dataset symlink mismatch'

# The original trainer creates files under encoder/checkpoint/lightgcn_plus and encoder/log/lightgcn_plus.
def link_output(source, destination):
    source.parent.mkdir(parents=True,exist_ok=True)
    destination.mkdir(parents=True,exist_ok=True)
    if source.is_symlink():
        if source.resolve()!=destination.resolve():
            source.unlink()
            source.symlink_to(destination,target_is_directory=True)
    elif source.exists():
        raise RuntimeError(f'Unexpected non-symlink output directory: {source}')
    else:
        source.symlink_to(destination,target_is_directory=True)

CHECKPOINT_DIR = OUTPUT_DIR/'checkpoints'
LOG_DIR = OUTPUT_DIR/'logs'
link_output(REPO/'encoder/checkpoint'/MODEL,CHECKPOINT_DIR)
link_output(REPO/'encoder/log'/MODEL,LOG_DIR)
print('CELL 2: PASS — source and E3-only output isolation')

In [ ]:
# CELL 3 — INHERIT E1 CONFIG WITHOUT CHANGING TRAINING HYPERPARAMETERS
import yaml
CONFIG_PATH = REPO/'encoder/config/modelconf/lightgcn_plus.yml'
with open(CONFIG_PATH,encoding='utf-8') as f: original_config = yaml.safe_load(f)
assert original_config['model']['name'] == MODEL
training_config = {
    **original_config,
    'train': {**original_config['train'], 'seed': SEED, 'save_model': True},
}
original_kd_weight = float(original_config['model']['kd_weight'])
if ALIGN_WEIGHT is not None:
    # Optional sensitivity, not part of 9 core runs.
    training_config['model'] = {**original_config['model'], 'kd_weight': float(ALIGN_WEIGHT)}
actual_weight = float(training_config['model']['kd_weight'])

run_info = dict(experiment='E3A_Semantic_Ablation', dataset=DATASET, model=MODEL,
    align_mode=ALIGN_MODE, seed=SEED, align_weight=actual_weight,
    align_weight_source=('E1_pinned_default' if ALIGN_WEIGHT is None else 'leader_override'),
    rlmrec_commit=RLMREC_COMMIT, training_config=training_config,
    pytorch_version=torch.__version__, python_version=sys.version.split()[0], status='prepared')
config_file = OUTPUT_DIR/'run_config.json'
# Safe overwrite of CONFIG only; finished run is protected in Cell 1
config_file.write_text(json.dumps(run_info,indent=2,default=str),encoding='utf-8')
print('KD weight E1 pinned:',original_kd_weight,'| used:',actual_weight)
print('E1 early stopping settings:',{k:training_config['train'].get(k) for k in ('epoch','patience','test_step')})
print('CELL 3: PASS')

In [ ]:
# CELL 4 — SAME E1 IMPORTS + BUILD HANDLER (CLEAN RUNTIME ONLY)
import random, numpy as np, pickle, inspect, ast, textwrap, types, importlib, traceback
from contextlib import redirect_stdout, redirect_stderr

os.chdir(REPO)
assert 'config.configurator' not in sys.modules, 'RLMRec config already imported. Runtime > Restart and Run All.'
original_argv = sys.argv[:]
sys.argv = ['train_encoder.py','--model',MODEL,'--dataset',DATASET,'--cuda','0','--seed',str(SEED)]
try:
    sys.path.insert(0,str(REPO/'encoder'))
    from config.configurator import configs
    from trainer.trainer import init_seed
    from data_utils.build_data_handler import build_data_handler
    from models.bulid_model import build_model
    from trainer.logger import Logger
    from trainer.build_trainer import build_trainer
finally:
    sys.argv = original_argv
configs['train'].update(training_config['train'])
configs['model'].update(training_config['model'])
assert configs['model']['name'] == MODEL
assert 20 in configs['test']['k'], 'E1 K=20 is required'
init_seed()
handler = build_data_handler()
handler.load_data()
print('CELL 4: PASS — inherited E1 data handler / config')

In [ ]:
# CELL 5 — ALIGNMENT SELECTOR BY SEMANTIC ROLE (ORDER-INDEPENDENT)
# WHY: ast.walk() is NOT execution order. Identify each InfoNCE term by its
# first-argument embedding name, not by whether the call is first/second/third.
# Keep original BPR, regularization, KD scaling, source checkpoint and trainer.
from collections import Counter
source_module = importlib.import_module('models.general_cf.lightgcn_plus')
SourceModel = getattr(source_module, 'LightGCN_plus')
_original_loss_method = SourceModel.cal_loss
_original_info = source_module.cal_infonce_loss
loss_source = textwrap.dedent(inspect.getsource(_original_loss_method))
parsed = ast.parse(loss_source)

ROLE_NAMES = ('anc_embeds', 'pos_embeds', 'neg_embeds')
info_calls = [node for node in ast.walk(parsed)
              if isinstance(node, ast.Call)
              and isinstance(node.func, ast.Name)
              and node.func.id == 'cal_infonce_loss']
assert len(info_calls) == 3, f'Expected exactly 3 InfoNCE calls, got {len(info_calls)}'
assert all(len(c.args) >= 3 and isinstance(c.args[0], ast.Name) for c in info_calls), (
    'InfoNCE source has changed; inspect pinned source before training')
call_names = [c.args[0].id for c in info_calls]
assert Counter(call_names) == Counter(ROLE_NAMES), (
    'Unexpected InfoNCE embedding roles: '+repr(call_names))
for pattern in ('cal_bpr_loss(', 'reg_params(self)', 'kd_loss /=', 'kd_loss *=', 'self.kd_weight'):
    assert pattern in loss_source, f'Unexpected pinned model loss: {pattern}'

ENABLED_BY_ROLE = {
    'graph_only': dict(anc_embeds=False, pos_embeds=False, neg_embeds=False),
    'user_only': dict(anc_embeds=True,  pos_embeds=False, neg_embeds=False),
    'item_only': dict(anc_embeds=False, pos_embeds=True,  neg_embeds=True),
    'full':      dict(anc_embeds=True,  pos_embeds=True,  neg_embeds=True),
}[ALIGN_MODE]
# Backward compatible with downstream verification cells (order = user, pos, neg).
ENABLED = tuple(ENABLED_BY_ROLE[name] for name in ROLE_NAMES)

def install_alignment_selector(model):
    def ablated_loss(self, batch_data):
        assert source_module.cal_infonce_loss is _original_info, 'Unexpected external InfoNCE patch'
        observed = []
        def switch_info(x, y, all_y, temp):
            # The caller is the ORIGINAL cal_loss. Its named tensors identify
            # the role unambiguously, independent of AST or runtime call order.
            frame = inspect.currentframe().f_back
            try:
                local_vars = frame.f_locals
                matches = [name for name in ROLE_NAMES
                           if name in local_vars and local_vars[name] is x]
            finally:
                del frame
            assert len(matches) == 1, (
                'Cannot uniquely identify InfoNCE role; refuse unsafe ablation: '
                +repr(matches))
            role = matches[0]
            assert role not in observed, f'InfoNCE role repeated: {role}'
            observed.append(role)
            if ENABLED_BY_ROLE[role]:
                return _original_info(x, y, all_y, temp)
            return x.sum() * 0.0  # differentiable zero; no semantic loss gradient
        source_module.cal_infonce_loss = switch_info
        try:
            result = _original_loss_method(self, batch_data)
        finally:
            source_module.cal_infonce_loss = _original_info
        assert Counter(observed) == Counter(ROLE_NAMES), (
            'Unexpected actual InfoNCE branches: '+repr(observed))
        self._e3_verified_calls = tuple(observed)
        return result
    model.cal_loss = types.MethodType(ablated_loss, model)
    return model

# Independent unit test: also verifies that the selection depends on role,
# not on the order in which the branch appears.
def selector_unit_test():
    for role_order in (ROLE_NAMES, tuple(reversed(ROLE_NAMES)),
                       ('neg_embeds', 'anc_embeds', 'pos_embeds')):
        dummy = torch.tensor([2.0], requires_grad=True)
        contrib = [dummy.sum()**2 if ENABLED_BY_ROLE[name] else dummy.sum()*0.0
                   for name in role_order]
        total = sum(contrib)
        total.backward()
        expected = sum(ENABLED_BY_ROLE.values())
        assert math.isclose(float(total.item()), 4*expected, rel_tol=1e-6)
        assert math.isclose(float(dummy.grad.item()), 4*expected, rel_tol=1e-6)
selector_unit_test()
print('InfoNCE names found by AST (order is irrelevant):',call_names)
print('Role-based selector:', dict(zip(('user','positive_item','negative_item'), ENABLED)))
print('CELL 5: PASS — all three roles verified, independent of call order')


In [ ]:
# CELL 6 — REAL MODEL GRADIENT SMOKE TEST (NO TRAINING SAVED)
import scipy.sparse as sp
# Capture RNG to prevent the diagnostic forward pass from changing the real initialization.
_rng = dict(py=random.getstate(), np=np.random.get_state(), cpu=torch.get_rng_state(),
            cuda=torch.cuda.get_rng_state_all())
smoke_model = install_alignment_selector(build_model(handler).to(configs['device']))
train_matrix = pickle.load(open(DATA_DIR/'trn_mat.pkl','rb')).tocsr()
assert train_matrix.shape == (smoke_model.user_num,smoke_model.item_num)
users, positives = train_matrix.nonzero()
assert len(users)>0
rr = np.random.default_rng(192837)
selected = rr.choice(len(users),size=min(8,len(users)),replace=False)
u = users[selected].astype(np.int64)
p = positives[selected].astype(np.int64)
ng=[]
for uid in u:
    tr_seen=set(train_matrix.indices[train_matrix.indptr[uid]:train_matrix.indptr[uid+1]])
    for _ in range(10000):
        candidate=int(rr.integers(0,smoke_model.item_num))
        if candidate not in tr_seen: break
    else: raise RuntimeError('No negative available for smoke user')
    ng.append(candidate)
triple = [torch.tensor(x,dtype=torch.long,device=configs['device']) for x in (u,p,np.array(ng))]
smoke_total, smoke_parts = smoke_model.cal_loss(triple)
assert torch.isfinite(smoke_total), 'Nonfinite training smoke loss'
assert all(torch.isfinite(x) for x in smoke_parts.values()), 'Nonfinite smoke loss part'
assert Counter(smoke_model._e3_verified_calls) == Counter(ROLE_NAMES)
smoke_total.backward()
assert smoke_model.user_embeds.grad is not None and torch.isfinite(smoke_model.user_embeds.grad).all()
assert smoke_model.item_embeds.grad is not None and torch.isfinite(smoke_model.item_embeds.grad).all()
smoke_results = {k:float(v.detach().cpu()) for k,v in smoke_parts.items()}
print('Actual-model smoke loss:',smoke_results)
print('Active terms:',sum(ENABLED),'Expected: ',sum(ENABLED))
del smoke_model
import gc
gc.collect(); torch.cuda.empty_cache()
random.setstate(_rng['py']); np.random.set_state(_rng['np']); torch.set_rng_state(_rng['cpu']); torch.cuda.set_rng_state_all(_rng['cuda'])
model = install_alignment_selector(build_model(handler).to(configs['device']))
logger = Logger()
trainer = build_trainer(handler,logger)
print('CELL 6: PASS — verified gradients and restored RNG for actual run')

In [ ]:
# ==========================================
# E3-A FULL TRAINING - CELL 4
# TRAIN + EARLY STOPPING + BEST CHECKPOINT
# ==========================================

import os
import sys
import json
import subprocess
import traceback
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr

# 1. Kiểm tra cấu hình từ Cell 3
config_file = OUTPUT_DIR / "run_config.json"
assert config_file.is_file(), "Run Cell 3 first!"

run_info = json.loads(config_file.read_text())

assert run_info["model"] == MODEL
assert run_info["seed"] == SEED

checkpoint_file = (
    CHECKPOINT_DIR /
    f"{MODEL}-{DATASET}-{SEED}.pth"
)

metrics_file = OUTPUT_DIR / "metrics.json"
history_file = OUTPUT_DIR / "validation_history.json"
console_file = OUTPUT_DIR / "training_console.log"

# Tránh vô tình ghi đè một lần chạy hoàn chỉnh
assert not metrics_file.exists(), (
    "This run already has metrics.json. "
    "Do not overwrite an existing experiment."
)

assert not checkpoint_file.exists(), (
    "Checkpoint already exists for this model/seed."
)

assert (
    REPO / "encoder/checkpoint" / MODEL
).resolve() == CHECKPOINT_DIR.resolve()

# 2-4. E1 imports, seed, handler and patched model were initialized by Cells 4–6.
assert MODEL == 'lightgcn_plus'
assert hasattr(model,'_e3_verified_calls') is False, 'Fresh training model should not have been used yet'
assert 20 in configs['test']['k']

# 5. Theo dõi validation & best epoch
history = []
best = {"epoch": None, "recall20": -1.0}
captured_test = {}
epoch_counter = {"completed": 0}

original_evaluate = trainer.evaluate
original_test = trainer.test
original_train_epoch = trainer.train_epoch

K_VALUES = configs["test"]["k"]

def format_metrics(raw):
    return {
        f"{metric}@{k}": float(value)
        for metric, values in raw.items()
        for k, value in zip(K_VALUES, values)
    }

def tracked_evaluate(current_model, epoch_idx=None):
    result = original_evaluate(current_model, epoch_idx)

    if epoch_idx is not None:
        values = format_metrics(result)
        recall20 = values["recall@20"]

        history.append({
            "epoch_index": int(epoch_idx),
            **values
        })

        if recall20 > best["recall20"]:
            best["epoch"] = int(epoch_idx)
            best["recall20"] = recall20

        history_file.write_text(
            json.dumps(history, indent=2),
            encoding="utf-8"
        )

    return result

def tracked_test(current_model):
    result = original_test(current_model)
    captured_test.update(format_metrics(result))
    return result

def tracked_train_epoch(current_model, epoch_idx):
    original_train_epoch(current_model, epoch_idx)
    epoch_counter["completed"] += 1

trainer.evaluate = tracked_evaluate
trainer.test = tracked_test
trainer.train_epoch = tracked_train_epoch

# 6. Ghi console ra cả notebook và Drive
class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()

run_info["status"] = "running"
config_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8"
)

print("=== E3-A FULL TRAINING ===")
print("Model:", MODEL)
print("Mode:", ALIGN_MODE)
print("Seed:", SEED)
print("Max epochs:", configs["train"]["epoch"])
print("Patience:", configs["train"]["patience"])
print("Checkpoint:", checkpoint_file)
print("Starting training...\n")

# 7. Full Training bằng trainer gốc
try:
    with open(console_file, "a", encoding="utf-8") as log:
        with redirect_stdout(Tee(sys.stdout, log)):
            with redirect_stderr(log):
                trainer.train(model)
except Exception:
    run_info["status"] = "failed"
    config_file.write_text(
        json.dumps(run_info, indent=2),
        encoding="utf-8"
    )
    print("TRAINING FAILED")
    print(traceback.format_exc())
    raise

# 8. Kiểm tra kết quả cuối cùng
assert checkpoint_file.is_file(), (
    "Training ended but checkpoint was not saved!"
)

assert checkpoint_file.stat().st_size > 0
assert captured_test, "Missing final test metrics!"
assert best["epoch"] is not None

# 9. Lưu metrics
result = {
    "experiment": "E3A_Semantic_Ablation",
    "align_mode": ALIGN_MODE,
    "align_weight": actual_weight,
    "model": MODEL,
    "dataset": DATASET,
    "seed": SEED,
    "rlmrec_commit": RLMREC_COMMIT,
    "epochs_completed": epoch_counter["completed"],
    "best_epoch_index": best["epoch"],
    "best_epoch_number": best["epoch"] + 1,
    "best_validation_recall20": best["recall20"],
    "test_metrics": captured_test,
    "checkpoint": str(checkpoint_file),
    "status": "completed"
}

metrics_file.write_text(
    json.dumps(result, indent=2),
    encoding="utf-8"
)

run_info["status"] = "completed"
run_info["epochs_completed"] = epoch_counter["completed"]
run_info["best_epoch_index"] = best["epoch"]

config_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8"
)

print("\n=== E3-A TRAINING COMPLETED ===")
print("Epochs completed:", result["epochs_completed"])
print("Best epoch:", result["best_epoch_number"])
print("Best validation Recall@20:",
      result["best_validation_recall20"])

print("\n=== FINAL TEST METRICS ===")
for metric, value in captured_test.items():
    print(f"{metric}: {value:.6f}")

print("\n=== SAVED FILES ===")
print("Metrics:", metrics_file)
print("Checkpoint:", checkpoint_file)
print("Validation history:", history_file)
print("Console log:", console_file)

print("\nCELL 4: SUCCESS")

In [ ]:
# ==========================================
# E1 FULL TRAINING - CELL 5
# VERIFY TRAINING RESULTS
# ==========================================

import json
import math
import torch
from pathlib import Path
from datetime import datetime, timezone

# 1. Xác định các file cần kiểm tra
config_file = OUTPUT_DIR / "run_config.json"
metrics_file = OUTPUT_DIR / "metrics.json"
history_file = OUTPUT_DIR / "validation_history.json"
console_file = OUTPUT_DIR / "training_console.log"

checkpoint_file = (
    CHECKPOINT_DIR / f"{MODEL}-{DATASET}-{SEED}.pth"
)

required_files = {
    "Run configuration": config_file,
    "Test metrics": metrics_file,
    "Validation history": history_file,
    "Training log": console_file,
    "Best checkpoint": checkpoint_file
}

print("=== E3A OUTPUT VERIFICATION ===")

# 2. Kiểm tra file tồn tại và không rỗng
for label, path in required_files.items():
    assert path.is_file(), f"Missing {label}: {path}"
    assert path.stat().st_size > 0, f"Empty {label}"

    print(f"[PASS] {label}")

# 3. Kiểm tra metadata
config = json.loads(config_file.read_text())
metrics = json.loads(metrics_file.read_text())
history = json.loads(history_file.read_text())

assert config["status"] == "completed"
assert metrics["status"] == "completed"
assert metrics["model"] == MODEL
assert metrics["align_mode"] == ALIGN_MODE
assert math.isclose(float(metrics["align_weight"]),actual_weight,rel_tol=0,abs_tol=1e-12)
assert metrics["seed"] == SEED
assert metrics["rlmrec_commit"] == RLMREC_COMMIT

assert metrics["epochs_completed"] >= 1
assert len(history) >= 1

print("[PASS] Run metadata")

# 4. Kiểm tra Recall / NDCG
test_metrics = metrics["test_metrics"]

required_metrics = [
    "recall@10",
    "recall@20",
    "ndcg@10",
    "ndcg@20"
]

for name in required_metrics:
    assert name in test_metrics, f"Missing metric: {name}"

    value = float(test_metrics[name])

    assert math.isfinite(value), f"Invalid metric: {name}"
    assert 0 <= value <= 1, f"Out-of-range metric: {name}"

assert test_metrics["recall@10"] <= test_metrics["recall@20"]

print("[PASS] Test metrics")

# 5. Kiểm tra validation history
best_recall = max(
    float(record["recall@20"])
    for record in history
)

assert math.isclose(
    best_recall,
    float(metrics["best_validation_recall20"]),
    rel_tol=1e-5,
    abs_tol=1e-7
)

assert any(
    record["epoch_index"] == metrics["best_epoch_index"]
    for record in history
)

print("[PASS] Best validation checkpoint selection")

# 6. Thử đọc checkpoint từ Google Drive
state_dict = torch.load(
    checkpoint_file,
    map_location="cpu",
    weights_only=True
)

assert isinstance(state_dict, dict)
assert len(state_dict) > 0

assert all(
    isinstance(value, torch.Tensor)
    for value in state_dict.values()
)

print("[PASS] Checkpoint can be loaded")

# 7. Kiểm tra log gốc
source_logs = list(LOG_DIR.glob("*.log"))

assert len(source_logs) >= 1, (
    "Missing original RLMRec training log"
)

print("[PASS] Original training logs")

# 8. Lưu báo cáo nghiệm thu
verification = {
    "experiment": "E3A_Semantic_Ablation",
    "model": MODEL,
    "align_mode": ALIGN_MODE,
    "align_weight": actual_weight,
    "enabled_alignment_terms": list(ENABLED),
    "smoke_test": smoke_results,
    "seed": SEED,
    "status": "passed",
    "verified_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "epochs_completed": metrics["epochs_completed"],
    "best_epoch": metrics["best_epoch_number"],
    "best_validation_recall20": best_recall,
    "test_metrics": {
        name: float(test_metrics[name])
        for name in required_metrics
    },
    "checkpoint_readable": True,
    "checkpoint_parameter_tensors": len(state_dict)
}

verification_file = OUTPUT_DIR / "verification.json"

verification_file.write_text(
    json.dumps(verification, indent=2),
    encoding="utf-8"
)

# 9. Kết quả cuối
print("\n=== FINAL RESULTS ===")
print("Model:", MODEL)
print("Seed:", SEED)
print("Epochs:", metrics["epochs_completed"])
print("Best epoch:", metrics["best_epoch_number"])

for name in required_metrics:
    print(f"{name}: {test_metrics[name]:.6f}")

print("\nVerification report:", verification_file)
print("\nE3A TRAINING VERIFICATION: PASS")

In [ ]:
# CELL 9 — EXPORT TRAIN-MASKED TOP-20 FROM VERIFIED BEST CHECKPOINT
# Export uses the saved best checkpoint, not a possibly later epoch.
# Only Train positives are masked; neither Validation nor Test labels are used for ranking.
import numpy as np
import scipy.sparse as sp
from datetime import datetime, timezone

ckpt = torch.load(checkpoint_file,map_location='cpu',weights_only=True)
assert isinstance(ckpt,dict) and ckpt
model.load_state_dict(ckpt,strict=True)
model.eval()
model.is_training=False
model.final_embeds=None

train_matrix = pickle.load(open(DATA_DIR/'trn_mat.pkl','rb')).tocsr()
test_matrix  = pickle.load(open(DATA_DIR/'tst_mat.pkl','rb')).tocsr()
assert train_matrix.shape == test_matrix.shape == (model.user_num,model.item_num)
test_users = np.flatnonzero(np.diff(test_matrix.indptr)>0).astype(np.int64)
assert len(test_users)==9860, f'Unexpected test-user count {len(test_users)}'
K=20
all_items=[]; all_scores=[]
with torch.no_grad():
    user_vecs, item_vecs = model.forward(model.adj,1.0)
    item_T=item_vecs.T
    for st in range(0,len(test_users),128):
        part=test_users[st:st+128]
        scores=user_vecs[torch.as_tensor(part,dtype=torch.long,device=user_vecs.device)]@item_T
        # Mask only training interactions: use CSR indices for each user in this batch.
        counts=np.diff(train_matrix.indptr[part].reshape(-1,1)) if False else None
        rr=[]; cc=[]
        for local_row,uid in enumerate(part):
            cs=train_matrix.indices[train_matrix.indptr[uid]:train_matrix.indptr[uid+1]]
            rr.extend([local_row]*len(cs)); cc.extend(cs.tolist())
        if rr:
            scores[torch.tensor(rr,dtype=torch.long,device=scores.device),
                   torch.tensor(cc,dtype=torch.long,device=scores.device)] = -torch.inf
        top_scores,top_items=torch.topk(scores,k=K,dim=1,largest=True,sorted=True)
        assert torch.isfinite(top_scores).all(), 'Insufficient eligible candidates'
        all_items.append(top_items.detach().cpu().numpy().astype(np.int32))
        all_scores.append(top_scores.detach().cpu().numpy().astype(np.float32))

pred_items=np.concatenate(all_items,axis=0)
pred_scores=np.concatenate(all_scores,axis=0)
assert pred_items.shape==(len(test_users),20)
assert pred_scores.shape==pred_items.shape
assert all(len(np.unique(r))==20 for r in pred_items), 'Duplicates in Top20'
for uid, recs in zip(test_users,pred_items):
    seen=train_matrix.indices[train_matrix.indptr[uid]:train_matrix.indptr[uid+1]]
    assert not np.isin(recs,seen).any(),f'Train leakage user={uid}'

# Independent metric check, same Test-user set and ranking convention as E1/E2.
computed={}
for k in (10,20):
    recalls=[]; ndcgs=[]
    for uid,recs in zip(test_users,pred_items):
        positives=test_matrix.indices[test_matrix.indptr[uid]:test_matrix.indptr[uid+1]]
        hits=np.isin(recs[:k],positives)
        recalls.append(hits.sum()/len(positives))
        discounts=1/np.log2(np.arange(2,k+2))
        ideal=discounts[:min(k,len(positives))].sum()
        ndcgs.append(float((hits*discounts).sum()/ideal))
    computed[f'recall@{k}']=float(np.mean(recalls))
    computed[f'ndcg@{k}']=float(np.mean(ndcgs))

# Stop on material mismatch. Helps catch evaluation convention differences before use in E2-style bias analysis.
max_dif=max(abs(computed[name]-float(metrics['test_metrics'][name])) for name in computed)
assert max_dif<1e-5, f'TopK accuracy mismatch vs trainer: {max_dif}; check evaluation and masking before accepting export. {computed} vs {metrics["test_metrics"]}'
pred_dir=OUTPUT_DIR/'topk'; pred_dir.mkdir(exist_ok=True)
pred_path=pred_dir/'topk_test.npz'
np.savez_compressed(pred_path,user_ids=test_users,item_ids=pred_items,scores=pred_scores)
export_metadata=dict(model=MODEL,align_mode=ALIGN_MODE,seed=SEED,k=20,
  source_checkpoint=str(checkpoint_file),rlmrec_commit=RLMREC_COMMIT,
  ranking_mask='train_only',validation_masked=False,test_labels_used_for_ranking=False,
  n_test_users=len(test_users),n_items=model.item_num,metrics_recalculated=computed,
  trainer_max_abs_difference=max_dif,exported_at_utc=datetime.now(timezone.utc).isoformat(),
  status='passed')
(pred_dir/'export_verification.json').write_text(json.dumps(export_metadata,indent=2),encoding='utf-8')
print('E3A EXPORT VERIFICATION: PASS')
print('TopK:',pred_path,'| shape:',pred_items.shape,'| max accuracy diff:',max_dif)

In [ ]:
# CELL 10 — FINAL STATUS (NO TRAINING)
assert (OUTPUT_DIR/'metrics.json').exists()
assert (OUTPUT_DIR/'verification.json').exists()
assert (OUTPUT_DIR/'topk/topk_test.npz').exists()
assert json.loads((OUTPUT_DIR/'verification.json').read_text())['status']=='passed'
assert json.loads((OUTPUT_DIR/'topk/export_verification.json').read_text())['status']=='passed'
report=dict(experiment='E3A',status='passed',align_mode=ALIGN_MODE,
    seed=SEED,align_weight=actual_weight,source_commit=RLMREC_COMMIT,
    infoNCE_components_enabled=dict(zip(['user','positive_item','negative_item'],ENABLED)),
    best_checkpoint=str(checkpoint_file),topk=str(OUTPUT_DIR/'topk/topk_test.npz'),
    validation_best_recall20=metrics['best_validation_recall20'],test_metrics=metrics['test_metrics'],
    full_ranking_mask='train_only',test_labels_used_for_ranking=False)
(OUTPUT_DIR/'e3a_verification.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('E3A_VERIFICATION: PASS')
print('MODE:',ALIGN_MODE,'SEED:',SEED,'OUTPUT:',OUTPUT_DIR)

## Tiếp theo sau E3-A

Chạy `graph_only`, `user_only`, `item_only` cho cả 3 seeds (nếu nhóm chọn thiết kế có đối chứng Graph-only nội bộ RLMRec-Con). `full` của E1 dùng để so sánh khi config tương đương, không cần huấn luyện lại mặc định. Sau khi có các `topk_test.npz`, cần chạy **E3 metrics aggregation** (Recall/NDCG và E2 popularity metrics) trên tất cả variants và tính **mean ± sample SD (`ddof=1`)**. E3-B agreement là notebook riêng.

**Giới hạn cần lưu ý:** Patch in-memory giữ nguyên hàm `reg_params(self)`; do đó cả Graph-only (trong kiến trúc Plus) cũng có regularization trên semantic MLP. Không đồng nhất nó với LightGCN E1, và cần ghi rõ khi viết phương pháp.